# 01 - Preparacao da CelebA-Spoof

Notebook para organizar o download da base, preparar as pastas locais e validar o manifesto JSON.

In [9]:
from pathlib import Path
import json
import tarfile
import zipfile
from itertools import islice

import pandas as pd

In [10]:
PROJECT_ROOT = Path.cwd().resolve().parent
DATASET_PARENT = PROJECT_ROOT / 'data' / 'raw'
DATASET_ROOT = DATASET_PARENT / 'CelebA-Spoof'

for folder in [
    PROJECT_ROOT / 'data',
    DATASET_PARENT,
    PROJECT_ROOT / 'artifacts',
    PROJECT_ROOT / 'reports',
    PROJECT_ROOT / 'reports' / 'figures',
]:
    folder.mkdir(parents=True, exist_ok=True)

DATASET_ROOT.mkdir(parents=True, exist_ok=True)
DATASET_ROOT

PosixPath('/home/arthur/Documentos/Github/Projeto-5-Redes-Neurais/data/raw/CelebA-Spoof')

## 1. Download oficial da base

Segundo o repositorio oficial da CelebA-Spoof, o dataset pode ser obtido em:

- Repositorio oficial: https://github.com/ZhangYuanhan-AI/CelebA-Spoof
- Google Drive: link listado no README oficial
- Baidu Drive: link listado no README oficial

A base e para pesquisa nao comercial e, na pratica, o download normalmente precisa ser feito manualmente no navegador.

## 2. Extracao opcional do arquivo baixado

Se voce ja baixou um arquivo compactado, ajuste `ARCHIVE_PATH`.

In [11]:
ARCHIVE_PATH = Path('../CelebA_Spoof.zip.001')
ARCHIVE_PATH

PosixPath('../CelebA_Spoof.zip.001')

In [12]:
if ARCHIVE_PATH.exists():
    if zipfile.is_zipfile(ARCHIVE_PATH):
        with zipfile.ZipFile(ARCHIVE_PATH, 'r') as archive:
            archive.extractall(DATASET_PARENT)
        print('ZIP extraido com sucesso em', DATASET_PARENT)
    elif tarfile.is_tarfile(ARCHIVE_PATH):
        with tarfile.open(ARCHIVE_PATH, 'r:*') as archive:
            archive.extractall(DATASET_PARENT)
        print('TAR extraido com sucesso em', DATASET_PARENT)
    else:
        print('Formato nao reconhecido automaticamente. Extraia manualmente para', DATASET_ROOT)
else:
    print('Defina ARCHIVE_PATH se quiser usar extracao automatica.')

Formato nao reconhecido automaticamente. Extraia manualmente para /home/arthur/Documentos/Github/Projeto-5-Redes-Neurais/data/raw/CelebA-Spoof


## 3. Verificacao rapida da estrutura

In [13]:
all_files = sorted(path.relative_to(DATASET_ROOT) for path in DATASET_ROOT.rglob('*'))
list(islice(all_files, 30))

[]

In [14]:
manifest_candidates = [
    DATASET_ROOT / 'metas' / 'intra_test' / 'train_label.json',
    DATASET_ROOT / 'metas' / 'intra_test' / 'test_label.json',
    DATASET_ROOT / 'train_label.json',
    DATASET_ROOT / 'test_label.json',
]

for candidate in manifest_candidates:
    print(candidate, '->', candidate.exists())

/home/arthur/Documentos/Github/Projeto-5-Redes-Neurais/data/raw/CelebA-Spoof/metas/intra_test/train_label.json -> False
/home/arthur/Documentos/Github/Projeto-5-Redes-Neurais/data/raw/CelebA-Spoof/metas/intra_test/test_label.json -> False
/home/arthur/Documentos/Github/Projeto-5-Redes-Neurais/data/raw/CelebA-Spoof/train_label.json -> False
/home/arthur/Documentos/Github/Projeto-5-Redes-Neurais/data/raw/CelebA-Spoof/test_label.json -> False


## 4. Funcoes locais para ler o manifesto

Estas funcoes ficam no proprio notebook para nao depender de arquivos `.py` externos.

In [15]:
SPLIT_FILES = {
    'train': 'train_label.json',
    'validation': 'test_label.json',
    'test': 'test_label.json',
}

def normalize_label(raw_label):
    if isinstance(raw_label, list) and raw_label:
        return int(raw_label[-1])
    return int(raw_label)

def load_manifest(dataset_root, split='train', max_samples=None):
    dataset_root = Path(dataset_root)
    split_file = SPLIT_FILES.get(split, f'{split}.json')
    manifest_path = dataset_root / 'metas' / 'intra_test' / split_file
    if not manifest_path.exists():
        manifest_path = dataset_root / split_file
    if not manifest_path.exists():
        raise FileNotFoundError(
            "Manifesto da CelebA-Spoof nao encontrado. "
            "Esperado em 'metas/intra_test/<split>_label.json' ou na raiz do dataset."
        )

    with manifest_path.open('r', encoding='utf-8') as file:
        data = json.load(file)

    rows = []
    for image_path, raw_metadata in data.items():
        if isinstance(raw_metadata, dict):
            label = raw_metadata.get('live_spoof', raw_metadata.get('label', 0))
            subject_id = raw_metadata.get('subject_id')
        else:
            label = raw_metadata
            subject_id = None

        rows.append({
            'image_path': image_path,
            'label': normalize_label(label),
            'subject_id': subject_id,
            'split': split,
        })

    frame = pd.DataFrame(rows)
    if max_samples:
        frame = frame.head(max_samples).copy()
    return frame

## 5. Validacao do manifesto

In [16]:
train_df = load_manifest(DATASET_ROOT, split='train', max_samples=10)
train_df.head()

FileNotFoundError: Manifesto da CelebA-Spoof nao encontrado. Esperado em 'metas/intra_test/<split>_label.json' ou na raiz do dataset.

In [ ]:
train_df['label'].value_counts(dropna=False)

Se esta etapa funcionar, a base ja esta pronta para os notebooks seguintes.